In [7]:
!git pull origin main
!pip install -r ../requirements.txt

From https://github.com/francesco-dorati/robotic_affordance_project
 * branch            main       -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.7 MB/s  0:00:01 eta 0:00:01


In [8]:
import subprocess
from pathlib import Path

# 1. Define URLs and Paths
umd_tools_url = "https://obj.umiacs.umd.edu/part-affordance/part-affordance-dataset-tools.tar.gz"
umd_clutter_url = "https://obj.umiacs.umd.edu/part-affordance/part-affordance-dataset-clutter.tar.gz"

# We stay consistent with your config.py paths
data_raw_dir = Path("../data/raw")
data_raw_dir.mkdir(parents=True, exist_ok=True)

def download_and_extract(url, target_dir, dataset_name):
    filename = url.split('/')[-1]
    file_path = target_dir / filename
    
    # --- Download ---
    if not file_path.exists():
        print(f"🚀 Downloading {filename}...")
        try:
            subprocess.run(["wget", "-c", url, "-O", str(file_path)], check=True)
        except Exception as e:
            print(f"❌ Error downloading {filename}: {e}")
            return
    else:
        print(f"✅ {filename} already exists. Skipping download.")

    # --- Extract ---
    extraction_check = target_dir / dataset_name
    if not extraction_check.exists():
        print(f"📦 Extracting {filename} to {target_dir}...")
        try:
            # -x: extract, -f: file, -z: gunzip, -C: target directory
            subprocess.run(["tar", "-xzf", str(file_path), "-C", str(target_dir)], check=True)
            print(f"✨ Extraction of {filename} complete!")
        except Exception as e:
            print(f"❌ Error extracting {filename}: {e}")
    else:
        print(f"✅ Data appears already extracted in {extraction_check}. Skipping.")

# 2. Execute for both Tools and Clutter
download_and_extract(umd_tools_url, data_raw_dir, "part-affordance-dataset")
download_and_extract(umd_clutter_url, data_raw_dir, "part-affordance-clutter")

# 3. Final Verification
print("\n--- Folder Structure Check ---")
!ls -R data/raw/part-affordance-dataset | head -n 5

✅ part-affordance-dataset-tools.tar.gz already exists. Skipping download.
✅ Data appears already extracted in ../data/raw/part-affordance-dataset. Skipping.
✅ part-affordance-dataset-clutter.tar.gz already exists. Skipping download.
✅ Data appears already extracted in ../data/raw/part-affordance-clutter. Skipping.

--- Folder Structure Check ---


ls: cannot access 'data/raw/part-affordance-dataset': No such file or directory


In [ ]:
# --- OPTION A: Start from scratch ---
!python ../scripts/train.py --epochs 25 --batch_size 8

# --- OPTION B: Resume from a local checkpoint ---
# !python ../scripts/train.py --resume --epochs 25 --batch_size 8

Device: cuda | Checkpoints: /home/desktop/robotic_affordance_project/checkpoints
Using cache found in /home/desktop/.cache/torch/hub/facebookresearch_dinov2_main
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.12.0+cu130)
    Python  3.10.19 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
/home/desktop/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/desktop/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/desktop/.cache/torch/hub/facebookresearch_dinov2

In [ ]:
# Plot curves and check for overfitting
!python ../scripts/visualize.py --history ../checkpoints/history.jsonl
# -> writes training_curves.png and training_summary.txt

# Detailed quantitative report
!python ../scripts/evaluate.py --checkpoint ../checkpoints/best.pth
# -> writes evaluation_val.json

# Visual sanity check on 8 predictions
!python ../scripts/visualize.py --history ../checkpoints/history.jsonl \
    --checkpoint checkpoints/best.pth --n_samples 8
# -> writes samples/000_<tool>.png ... 007_<tool>.png